
本 notebook 演示如何使用 OpenAI 和 [MongoDB Atlas vector search](https://www.mongodb.com/products/platform/atlas-vector-search) 构建语义搜索应用

In [ ]:
!pip install pymongo openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 677.1/677.1 kB 10.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.3/220.3 kB 24.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 29.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 9.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.9/76.9 kB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 6.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llmx 0.0.15a0 requires cohere, which is not installed.
llmx 0.0.15a0 requires tiktoken, which is not installed.


# 第 1 步：设置环境

这需要满足 2 个前提条件：

1.   **MongoDB Atlas 集群**：要创建一个永久免费的 MongoDB Atlas 集群，如果你还没有 MongoDB Atlas 账户，首先需要创建一个。访问 [MongoDB Atlas 网站](https://www.mongodb.com/atlas/database)并点击“Register”。访问 [MongoDB Atlas](https://account.mongodb.com/account/login) 仪表盘并设置你的集群。要在聚合管道中使用 `$vectorSearch` 运算符，你需要运行 MongoDB Atlas 6.0.11 或更高版本。本教程可以使用免费集群完成。设置部署时，系统会提示你设置数据库用户和网络连接规则。请务必将用户名和密码保存在安全的地方，并配置正确的 IP 地址规则，以便集群能够正常连接。如果你需要更多入门帮助，请查看我们的 [MongoDB Atlas 教程](https://www.mongodb.com/basics/mongodb-atlas-tutorial)。

2. **OpenAI API 密钥**   要创建 OpenAI 密钥，你需要先创建一个账户。之后访问 [OpenAI 平台](https://platform.openai.com/)。点击屏幕右上角的个人资料图标打开下拉菜单，然后选择“View API keys”。


In [ ]:
import getpass

MONGODB_ATLAS_CLUSTER_URI = getpass.getpass("MongoDB Atlas Cluster URI:")
OPENAI_API_KEY = getpass.getpass("OpenAI API Key:")


MongoDB Atlas Cluster URI:··········
OpenAI API Key:··········


注意：执行上述步骤后，系统会提示你输入凭据。

在本教程中，我们将使用
[MongoDB 示例数据集](https://www.mongodb.com/docs/atlas/sample-data/)。使用 Atlas UI 加载示例数据集。我们将使用“sample_mflix”数据库，其中包含一个“movies”集合，集合中的每个文档都包含 title、plot、genres、cast、directors 等字段。


In [ ]:
import openai
import pymongo

client = pymongo.MongoClient(MONGODB_ATLAS_CLUSTER_URI)
db = client.sample_mflix
collection = db.movies

openai.api_key = OPENAI_API_KEY

In [ ]:
ATLAS_VECTOR_SEARCH_INDEX_NAME = "default"
EMBEDDING_FIELD_NAME = "embedding_openai_nov19_23"

# 第 2 步：设置嵌入生成函数

In [ ]:
model = "text-embedding-3-small"
def generate_embedding(text: str) -> list[float]:
    return openai.embeddings.create(input = [text], model=model).data[0].embedding


# 第 3 步：创建并存储嵌入

示例数据集 sample_mflix.movies 中的每个文档对应一部电影；我们将执行一个操作，为 "plot" 字段中的数据创建向量嵌入并将其存储到数据库中。要基于意图执行相似性搜索，就必须使用 OpenAI 嵌入端点创建向量嵌入。

In [ ]:
from pymongo import ReplaceOne

# Update the collection with the embeddings
requests = []

for doc in collection.find({'plot':{"$exists": True}}).limit(500):
  doc[EMBEDDING_FIELD_NAME] = generate_embedding(doc['plot'])
  requests.append(ReplaceOne({'_id': doc['_id']}, doc))

collection.bulk_write(requests)

BulkWriteResult({'writeErrors': [], 'writeConcernErrors': [], 'nInserted': 0, 'nUpserted': 0, 'nMatched': 50, 'nModified': 50, 'nRemoved': 0, 'upserted': []}, acknowledged=True)

执行上述操作后，"movies" 集合中的文档除了已有的 title、plot、genres、cast、directors 等字段外，还会多出一个由 `EMBEDDING_FIELD_NAME` 变量定义的 "embedding" 字段。

注意：为了节省时间，我们只处理 500 个文档。如果你想对 sample_mflix 数据库中全部 23,000 多个文档执行此操作，会需要一些时间。或者，你可以使用 [sample_mflix.embedded_movies 集合](https://www.mongodb.com/docs/atlas/sample-data/sample-mflix/#sample_mflix.embedded_movies)，其中包含预先填充的 `plot_embedding` 字段，该字段存有使用 OpenAI 的 `text-embedding-3-small` 嵌入模型创建的嵌入，可直接与 Atlas Search 的向量搜索功能配合使用。




# 第 4 步：创建向量搜索索引

我们将在这个集合上创建 Atlas Vector Search 索引，以便执行近似 KNN 搜索，为语义搜索提供支持。
我们将介绍创建该索引的 2 种方式——使用 Atlas UI 和使用 MongoDB python 驱动。

（可选）[文档：创建向量搜索索引 ](https://www.mongodb.com/docs/atlas/atlas-search/field-types/knn-vector/)

现在前往 [Atlas UI](https://cloud.mongodb.com)，按照[这里](https://www.mongodb.com/docs/atlas/atlas-vector-search/vector-search-tutorial/#create-the-atlas-vector-search-index)描述的步骤创建 Atlas Vector Search 索引。值为 1536 的 'dimensions' 字段对应 OpenAI text-embedding-ada-002。

在 Atlas UI 的 JSON 编辑器中使用下面给出的定义。

```
{
  "mappings": {
    "dynamic": true,
    "fields": {
      "embedding": {
        "dimensions": 1536,
        "similarity": "dotProduct",
        "type": "knnVector"
      }
    }
  }
}
```

（可选）或者，我们也可以使用 [pymongo 驱动以编程方式创建这些向量搜索索引](https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.create_search_index)
下面单元格中给出的 python 命令将创建该索引（仅适用于最新版本的 MongoDB Python 驱动以及服务器版本为 7.0+ 的 MongoDB Atlas 集群）。

In [ ]:
collection.create_search_index(
    {"definition":
        {"mappings": {"dynamic": True, "fields": {
            EMBEDDING_FIELD_NAME : {
                "dimensions": 1536,
                "similarity": "dotProduct",
                "type": "knnVector"
                }}}},
     "name": ATLAS_VECTOR_SEARCH_INDEX_NAME
    }
)

'default'

# 第 5 步：查询数据

这里的查询结果会找出情节与查询字符串中文本在语义上相似的电影，而不是基于关键词搜索。

（可选）[文档：运行向量搜索查询](https://www.mongodb.com/docs/atlas/atlas-vector-search/vector-search-stage/)

In [ ]:

def query_results(query, k):
  results = collection.aggregate([
    {
        '$vectorSearch': {
            "index": ATLAS_VECTOR_SEARCH_INDEX_NAME,
            "path": EMBEDDING_FIELD_NAME,
            "queryVector": generate_embedding(query),
            "numCandidates": 50,
            "limit": 5,
        }
    }
    ])
  return results

In [ ]:
query="imaginary characters from outerspace at war with earthlings"
movies = query_results(query, 5)

for movie in movies:
    print(f'Movie Name: {movie["title"]},\nMovie Plot: {movie["plot"]}\n')